# Lösning: Labb 02 - En tsunami som närmar sig stranden

Detta är en förenklad svensk lösningsnotebook till `labs/02_tsunami_shoaling_sv.md`.

Tanken är en 30-minutersaktivitet för ungefär 10-åriga barn: mycket att titta på, mycket att gissa, nästan ingen matematik. Läraren kör koden och pausar vid bilderna.

Vi vill visa en enda huvudidé: **en lång våg går fort på djupt vatten, bromsar på grunt vatten och kan då bli högre**.


## 1. Förberedelser

Vi importerar bara det som behövs. Animationerna sparas i `animations/`, som redan är en genererad mapp i projektet.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation

# Gör src/ importerbar även om notebooken körs från notebooks/.
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / "src").exists() else HERE.parent
SRC_DIR = PROJECT_ROOT / "src"
ANIMATION_DIR = PROJECT_ROOT / "animations"
ANIMATION_DIR.mkdir(exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from shallowwater import (
    ModelParams,
    make_grid,
    zero_forcing,
    compute_dt_cfl,
    run_model,
    shelf_bathymetry,
    make_sponge_hook,
)

plt.rcParams["figure.dpi"] = 120

# Sätt till False om du bara vill visa animationerna i notebooken.
SPARA_ANIMATIONER = True


## 2. Några ord innan modellen

Börja gärna med bilden nedan. Den är inte en tsunami, bara ett sätt att peka ut orden **vågtopp** och **våglängd**.

En viktig mening för barnen: vattnet åker inte som en bil hela vägen över havet. Det är **vågformen** som skickas vidare.


In [ ]:
x_demo = np.linspace(0, 12, 500)
y_demo = 0.55 * np.sin(2 * np.pi * (x_demo - 1.5) / 6)

fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(x_demo, y_demo, color="#1261a0", linewidth=3)
ax.axhline(0, color="0.55", linewidth=1)
ax.scatter([3, 9], [0.55, 0.55], color="#c2410c", zorder=3)
ax.text(3, 0.72, "vågtopp", ha="center", fontsize=11)
ax.text(9, 0.72, "vågtopp", ha="center", fontsize=11)
ax.annotate("", xy=(9, -0.78), xytext=(3, -0.78), arrowprops={"arrowstyle": "<->", "linewidth": 2})
ax.text(6, -1.02, "våglängd", ha="center", fontsize=12)
ax.set_ylim(-1.2, 1.05)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("En våg har toppar och en våglängd")
ax.spines[["left", "right", "top", "bottom"]].set_visible(False)
plt.show()


## 3. Ett större modellhav

Modellhavet är ungefär 4200 km långt. Till vänster finns djupt hav. Till höger finns en grundare kust. Botten lutar mjukt upp mot kusten.


In [ ]:
Nx, Ny = 220, 60
Lx, Ly = 4_200e3, 900e3

grid = make_grid(Nx, Ny, Lx, Ly)

H_deep = 4000.0       # djupt hav [m]
H_coast = 60.0        # grunt kustvatten [m]
shelf_width = 1_100e3 # bredd på den sluttande botten [m]

H = shelf_bathymetry(
    grid,
    H_deep=H_deep,
    H_coast=H_coast,
    shelf_width=shelf_width,
    coast="east",
    power=1.25,
)

params = ModelParams(H=H, f0=0.0, beta=0.0, r=0.0, linear=True)

x_km = grid.x_c / 1000
y_km = grid.y_c / 1000
X_km, Y_km = np.meshgrid(x_km, y_km)
center_j = Ny // 2
grund_start_km = (grid.Lx - shelf_width) / 1000


In [ ]:
botten_km = -H[center_j, :] / 1000

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.fill_between(x_km, botten_km, -4.3, color="#7a6a58", alpha=0.9, label="havsbotten")
ax.fill_between(x_km, 0, botten_km, color="#9bd4f0", alpha=0.45, label="vatten")
ax.plot(x_km, botten_km, color="#4a3528", linewidth=2)
ax.axvspan(grund_start_km, grid.Lx / 1000, color="#facc15", alpha=0.18, label="grundare område")
ax.axhline(0, color="#0f5e9c", linewidth=1)
ax.text(250, -0.35, "djupt hav", fontsize=11)
ax.text(grid.Lx / 1000 - 520, -0.35, "kust", fontsize=11)
ax.set_xlim(0, grid.Lx / 1000)
ax.set_ylim(-4.3, 0.35)
ax.set_xlabel("avstånd österut [km]")
ax.set_ylabel("djup under ytan [km]")
ax.set_title("Botten lutar upp mot kusten")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.25)
plt.show()


## 4. Prata om fart utan formler

För barnen räcker jämförelsen: ute på djupt hav kan en tsunami röra sig ungefär som ett flygplan. När havet blir grundare bromsas den kraftigt.

Säg gärna: **den främre delen av vågen bromsar först, medan resten av vågen fortfarande kommer bakifrån**.


In [ ]:
def ungefarlig_langvagsfart_kmh(djup_meter):
    return np.sqrt(params.g * djup_meter) * 3.6

fart_djupt = ungefarlig_langvagsfart_kmh(H_deep)
fart_kust = ungefarlig_langvagsfart_kmh(H_coast)

fig, ax = plt.subplots(figsize=(7.5, 2.6))
namn = ["djupt hav", "nära kusten"]
fart = [fart_djupt, fart_kust]
farger = ["#2563eb", "#f97316"]
ax.barh(namn, fart, color=farger)
ax.set_xlim(0, 800)
ax.set_xlabel("ungefärlig fart [km/h]")
ax.set_title("Samma sorts långa våg kan gå mycket olika fort")
for i, v in enumerate(fart):
    ax.text(v + 15, i, f"ca {v:.0f} km/h", va="center")
ax.grid(True, axis="x", alpha=0.25)
plt.show()

print(f"På djupt hav: ungefär {fart_djupt:.0f} km/h, som ett flygplan.")
print(f"Nära kusten: ungefär {fart_kust:.0f} km/h, mycket långsammare.")


## 5. Starta en låg, bred våg

Startvågen är låg men väldigt bred. I verkligheten kan en tsunami ha en enorm våglängd. Här gör vi en enkel vågform som börjar ute på djupt vatten.


In [ ]:
def tsunami_initial_condition(grid, params):
    X, Y = np.meshgrid(grid.x_c, grid.y_c)
    x0 = 650e3
    y0 = 0.5 * grid.Ly
    bredd_x = 170e3
    bredd_y = 0.42 * grid.Ly
    amp = 0.18

    eta = amp * np.exp(-((X - x0) ** 2) / (2 * bredd_x ** 2))
    eta *= np.exp(-((Y - y0) ** 2) / (2 * bredd_y ** 2))
    u = np.zeros((grid.Ny, grid.Nx + 1))
    v = np.zeros((grid.Ny + 1, grid.Nx))
    return eta, u, v

eta0, u0, v0 = tsunami_initial_condition(grid, params)

fig, ax = plt.subplots(figsize=(9, 3.2))
im = ax.imshow(
    eta0,
    origin="lower",
    extent=[0, grid.Lx / 1000, 0, grid.Ly / 1000],
    aspect="auto",
    cmap="RdBu_r",
    vmin=-0.2,
    vmax=0.2,
)
ax.contour(X_km, Y_km, H / 1000, levels=[0.1, 0.5, 1, 2, 3], colors="k", alpha=0.2, linewidths=0.7)
ax.axvspan(grund_start_km, grid.Lx / 1000, color="#facc15", alpha=0.14)
ax.set_xlabel("avstånd österut [km]")
ax.set_ylabel("avstånd nord-syd [km]")
ax.set_title("Startläge: en låg våg ute på djupt hav")
fig.colorbar(im, ax=ax, label="vattenytans höjd [m]")
plt.show()


## 6. Kör modellen

Koden nedan låter vågen röra sig i några timmar i modellhavet. En mjuk kant längst till vänster dämpar den del av vågen som går åt fel håll, så att lektionen kan fokusera på kusten till höger.


In [ ]:
import contextlib
import io

sponge_width = 450e3
sponge = make_sponge_hook(width=sponge_width, tau=1200.0, sides=("west",), power=2.0)

dt = compute_dt_cfl(grid, params, cfl=0.45)
tmax = 7.0 * 3600.0
save_every = max(1, int((8 * 60) / dt))

print(f"Modellen tar tidssteg på ungefär {dt:.0f} sekunder.")
print("Vi sparar en bild ungefär var åttonde minut.")
print("Nu får vågen resa genom modellhavet...")

teknisk_logg = io.StringIO()
with contextlib.redirect_stdout(teknisk_logg):
    out = run_model(
        tmax,
        dt,
        grid,
        params,
        zero_forcing,
        tsunami_initial_condition,
        save_every=save_every,
        hooks=[sponge],
        show_progress=False,
    )

print(f"Klart. Antal sparade bilder: {len(out['time'])}")


## 7. Hjälpfunktioner för animationer

De här funktionerna finns bara för att göra svenska, tydliga animationer. De förstorar vattenytans rörelse så att barnen faktiskt kan se vågen.


In [ ]:
def stapla_vattenyta(out):
    return np.stack(out["eta"], axis=0)

def valj_ramar(out, max_antal):
    antal = len(out["eta"])
    return np.unique(np.linspace(0, antal - 1, min(antal, max_antal), dtype=int))

def tid_text(sekunder):
    return f"{sekunder / 3600:.1f} timmar"

class SvenskAnimation:
    def __init__(self, anim, fig):
        self.animation = anim
        self.figure = fig

    def _repr_html_(self):
        return self.animation.to_jshtml()

    def save(self, path, fps=8, dpi=130):
        self.animation.save(path, writer="pillow", fps=fps, dpi=dpi)

def animera_uppifran(out, grid, H, max_ramar=55, interval=130):
    eta = stapla_vattenyta(out)
    tider = np.asarray(out["time"])
    ramar = valj_ramar(out, max_ramar)
    eta_sel = eta[ramar]
    vmax = float(np.nanpercentile(np.abs(eta_sel), 99))
    vmax = max(vmax, 1e-6)

    fig, ax = plt.subplots(figsize=(9, 3.4))
    im = ax.imshow(
        eta_sel[0],
        origin="lower",
        extent=[0, grid.Lx / 1000, 0, grid.Ly / 1000],
        aspect="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.contour(X_km, Y_km, H / 1000, levels=[0.1, 0.5, 1, 2, 3], colors="k", alpha=0.18, linewidths=0.7)
    ax.axvspan(grund_start_km, grid.Lx / 1000, color="#facc15", alpha=0.12)
    ax.set_xlabel("avstånd österut [km]")
    ax.set_ylabel("avstånd nord-syd [km]")
    fig.colorbar(im, ax=ax, label="vattenytans höjd [m]")

    def update(k):
        im.set_data(eta_sel[k])
        ax.set_title(f"Vågen sedd uppifrån i 2D - tid {tid_text(tider[ramar[k]])}")
        return (im,)

    update(0)
    anim = animation.FuncAnimation(fig, update, frames=len(ramar), interval=interval, blit=False)
    return SvenskAnimation(anim, fig)

def animera_sidovy(out, grid, H, max_ramar=60, interval=130, vag_forstoring=8000):
    eta = stapla_vattenyta(out)
    tider = np.asarray(out["time"])
    ramar = valj_ramar(out, max_ramar)
    botten = -H[center_j, :] / 1000
    golv = -4.35

    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.fill_between(x_km, botten, golv, color="#7a6a58", alpha=0.9)
    ax.plot(x_km, botten, color="#4a3528", linewidth=1.8)
    ax.axhline(0, color="0.35", linewidth=0.8, alpha=0.7)
    ax.axvspan(grund_start_km, grid.Lx / 1000, color="#facc15", alpha=0.14)

    vatten_fyllning = None
    vattenlinje, = ax.plot([], [], color="#0f5e9c", linewidth=2.4)

    ax.set_xlim(0, grid.Lx / 1000)
    ax.set_ylim(golv, 2.1)
    ax.set_xlabel("avstånd österut [km]")
    ax.set_ylabel("höjd i bilden [km]")
    ax.grid(True, alpha=0.25)
    ax.text(60, 1.65, f"Vågens höjd är förstorad {vag_forstoring} gånger", fontsize=10)

    def update(k):
        nonlocal vatten_fyllning
        if vatten_fyllning is not None:
            vatten_fyllning.remove()
        vatten = eta[ramar[k], center_j, :] * vag_forstoring / 1000
        vatten_fyllning = ax.fill_between(x_km, botten, vatten, color="#9bd4f0", alpha=0.42)
        vattenlinje.set_data(x_km, vatten)
        ax.set_title(f"Vågen sedd från sidan i 2D - tid {tid_text(tider[ramar[k]])}")
        return (vattenlinje, vatten_fyllning)

    update(0)
    anim = animation.FuncAnimation(fig, update, frames=len(ramar), interval=interval, blit=False)
    return SvenskAnimation(anim, fig)

def animera_3d(out, grid, H, max_ramar=34, interval=170, vag_forstoring=8000, steg_x=4, steg_y=3):
    eta = stapla_vattenyta(out)
    tider = np.asarray(out["time"])
    ramar = valj_ramar(out, max_ramar)

    xs = slice(None, None, steg_x)
    ys = slice(None, None, steg_y)
    X3, Y3 = np.meshgrid(grid.x_c[xs] / 1000, grid.y_c[ys] / 1000)
    botten = -H[ys, xs] / 1000
    eta_sel = eta[ramar, ys, xs]
    vmax = float(np.nanpercentile(np.abs(eta_sel), 99))
    zmax = max(1.8, 1.25 * vmax * vag_forstoring / 1000)

    fig = plt.figure(figsize=(9, 5.2))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(X3, Y3, botten, color="#7a6a58", alpha=0.58, linewidth=0, antialiased=False)
    ax.set_xlim(0, grid.Lx / 1000)
    ax.set_ylim(0, grid.Ly / 1000)
    ax.set_zlim(-4.4, zmax)
    ax.set_xlabel("öst [km]")
    ax.set_ylabel("nord-syd [km]")
    ax.set_zlabel("höjd i bilden [km]")
    ax.view_init(elev=28, azim=-63)
    ax.set_box_aspect((4.2, 0.9, 0.8))
    ax.text2D(0.02, 0.95, f"Vågen är förstorad {vag_forstoring} gånger", transform=ax.transAxes)

    vattenyta = None

    def update(k):
        nonlocal vattenyta
        if vattenyta is not None:
            vattenyta.remove()
        vatten = eta_sel[k] * vag_forstoring / 1000
        vattenyta = ax.plot_surface(
            X3,
            Y3,
            vatten,
            cmap="RdBu_r",
            vmin=-vmax * vag_forstoring / 1000,
            vmax=vmax * vag_forstoring / 1000,
            alpha=0.86,
            linewidth=0,
            antialiased=False,
        )
        ax.set_title(f"3D: vattenyta och sluttande botten - tid {tid_text(tider[ramar[k]])}")
        return (vattenyta,)

    update(0)
    anim = animation.FuncAnimation(fig, update, frames=len(ramar), interval=interval, blit=False)
    return SvenskAnimation(anim, fig)


## 8. Första animationen: uppifrån i 2D

Börja här med barnen. Be dem följa vågen med fingret: var går den fort, och var ändrar den form?


In [ ]:
anim_uppifran = animera_uppifran(out, grid, H)

if SPARA_ANIMATIONER:
    anim_uppifran.save(str(ANIMATION_DIR / "02_tsunami_shoaling_sv_2d_uppifran.gif"), fps=8)

anim_uppifran


## 9. Andra animationen: vågen från sidan i 2D

Nu syns botten under vågen. Detta är ofta den mest didaktiska bilden: vågen kommer från djupt vatten och möter en botten som stiger.

Kom ihåg att vattenytans rörelse är starkt förstorad. Annars skulle vågen nästan inte synas.


In [ ]:
anim_sidovy = animera_sidovy(out, grid, H)

if SPARA_ANIMATIONER:
    anim_sidovy.save(str(ANIMATION_DIR / "02_tsunami_shoaling_sv_2d_sidovy.gif"), fps=8)

anim_sidovy


## 10. Till sist: 3D med botten under

3D-bilden kommer sist, när barnen redan har förstått 2D-bilderna. Den visar den sluttande botten under vattenytan.

Den är inte i rätt skala: vågen är mycket mer uppförstorad än den skulle vara i verkligheten. Poängen är att se **sambandet mellan botten och vågen**.


In [ ]:
anim_3d = animera_3d(out, grid, H)

if SPARA_ANIMATIONER:
    anim_3d.save(str(ANIMATION_DIR / "02_tsunami_shoaling_sv_3d_botten.gif"), fps=7)

anim_3d


## 11. Stoppbilder för samtal

Om tiden är kort kan man hoppa över långa förklaringar och bara använda fyra stoppbilder: start, på djupt hav, över sluttningen och nära kusten.


In [ ]:
eta_stack = stapla_vattenyta(out)
tider = np.asarray(out["time"])
valda = np.unique(np.linspace(0, len(out["eta"]) - 1, 4, dtype=int))
vmax = float(np.nanpercentile(np.abs(eta_stack[valda]), 99))
vmax = max(vmax, 1e-6)

fig, axes = plt.subplots(1, len(valda), figsize=(12, 2.8), sharex=True, sharey=True)
for ax, k in zip(axes, valda):
    im = ax.imshow(
        eta_stack[k],
        origin="lower",
        extent=[0, grid.Lx / 1000, 0, grid.Ly / 1000],
        aspect="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.axvspan(grund_start_km, grid.Lx / 1000, color="#facc15", alpha=0.12)
    ax.set_title(tid_text(tider[k]))
    ax.set_xlabel("km")
axes[0].set_ylabel("km")
fig.suptitle("Fyra stoppbilder av vågens resa")
fig.colorbar(im, ax=axes, label="vattenytans höjd [m]", shrink=0.8)
plt.show()


## 12. Sammanfattande facit för läraren

Bra svar från barnen kan låta så här:

- Havet är djupast till vänster och grundast nära kusten till höger.
- Vågen går snabbast på djupt vatten.
- När vågen kommer in på grundare vatten bromsar den och kan bli högre.
- Våglängd är avståndet mellan två vågtoppar.
- 3D-bilden hjälper oss att se botten under vågen, men höjden är överdriven.

Den viktigaste slutmeningen är: **botten under havet kan ändra hur en våg rör sig och hur hög den blir**.
